# 03 · Small SFT: trilingual writing assistant (GPU, 1×T4 is enough)

**Inputs:** outputs of `01_tokenizer_and_data` and `02_pretrain`. Builds the SFT mix (CohereLabs/aya_dataset filtered to eng/spa/fra, Apache-2.0, plus self-supervised "continue this text" tasks in each language) and fine-tunes the pretrained model with loss on the replies only.

In [ ]:
# --- Get the Trilingua code -------------------------------------------------------
# Option A (after the repo is public): clone it.
# Option B: upload the repo folder as a Kaggle Dataset named "trilingua-code" and attach it.
import os, sys, glob, shutil, subprocess
REPO_URL = "https://github.com/aladewealth/Trilingual-.git"
CODE = "/tmp/trilingua"
attached = glob.glob("/kaggle/input/**/trilingua/model.py", recursive=True)
if os.path.exists(CODE):
    pass
elif attached:
    shutil.copytree(os.path.dirname(os.path.dirname(attached[0])), CODE)
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, CODE], check=True)
os.chdir(CODE); sys.path.insert(0, CODE)
print("code at", CODE, os.listdir(CODE))

In [ ]:
# Locate the data produced by 01_tokenizer_and_data (attach that notebook's output as input)
metas = glob.glob("/kaggle/input/**/meta.json", recursive=True)
assert metas, "Attach the output of 01_tokenizer_and_data as an input dataset"
DATA = os.path.dirname(metas[0])
print("DATA =", DATA); print(open(os.path.join(DATA, "meta.json")).read())
from trilingua.utils import find_latest_checkpoint
BASE = find_latest_checkpoint(['/kaggle/input'])
print('base checkpoint:', BASE)

In [ ]:
!python -m trilingua.sft build --data_dir /kaggle/working/sft_data --cont_per_lang 1500
!cat /kaggle/working/sft_data/sft_stats.json

In [ ]:
!python -m trilingua.sft train --data_dir /kaggle/working/sft_data --tokenizer {DATA}/tokenizer.json \
    --init_ckpt {BASE} --out_dir /kaggle/working/sft_out --epochs 2 --micro_batch 8 --grad_accum 4 --lr 1e-4

In [ ]:
from trilingua.inference import load_bundle, build_prompt, stream_text
b = load_bundle('/kaggle/working/sft_out/sft_final.pt', tokenizer=f'{DATA}/tokenizer.json', device='cuda')
for lang, topic in [('en', 'the ocean'), ('es', 'el océano'), ('fr', "l'océan")]:
    out = list(stream_text(b, build_prompt(b, lang, 'write', topic), 120, seed=0))[-1]
    print(f'[{lang}]', out, '\n')